# Notebook 7 | Law of Total Probability

In [1]:
from foundations_of_probability_and_statistics.cars.car_distribution import create_joint_car_distribution

import pandas as pd

# show all rows of data frames and series per default
pd.set_option("display.max_rows", None)

## The law of total probability: splitting across a partition

How do we compute the probability of one event by splitting it into disjoint cases? Let $A_1, \dots, A_m$ be a partition of the sample space -- disjoint events covering everything. Then any event $C = c$ decomposes as, in full form first and then in shorthand with dummy index $b$:

$$p(C = c) = \sum_b p(B = b) \, p(C = c \mid B = b), \qquad p(c) = \sum_b p(b) \, p(c \mid b).$$

The brands partition the cars ($A_b = \{B = b\}$), so the law applies directly. This is marginalization (notebook 2) with the product rule (notebook 5) substituted in -- and it becomes the denominator of Bayes' theorem in notebook 9, so learn to spot it there.

## Derivation

1. Split $\{C = c\}$ by brand: $\{C = c\} = \bigcup_b \{B = b, C = c\}$, a disjoint union -- no car belongs to two brands.
2. Add the piece probabilities: $p(c) = \sum_b p(b, c)$ (finite additivity, i.e. the sum rule of notebook 2).
3. Apply the product rule to each piece: $p(b, c) = p(b) \, p(c \mid b)$.
4. Substitute back to get $p(c) = \sum_b p(b) \, p(c \mid b)$.

The partitioning step is the one to check: with overlapping cases the sum double-counts, with a gappy cover it under-counts. Brands are safe on both counts.

## Worked example (by hand)

$$p(\text{black}) = 0.5 \cdot 0.3 + 0.3 \cdot 0.4 + 0.2 \cdot 0.2 = 0.31,$$
$$p(\text{blue}) = 0.5 \cdot 0.2 + 0.3 \cdot 0.2 + 0.2 \cdot 0 = 0.16,$$

where Ferrari never comes in blue, so $p(\text{blue} \mid \text{Ferrari}) = 0$ contributes a zero term -- kept explicit so the partition stays complete. Black is the majority paint precisely because two large brands favor it.

In [2]:
import math

joint = create_joint_car_distribution()

# law of total probability over the brand partition
p_black = 0.5 * 0.3 + 0.3 * 0.4 + 0.2 * 0.2
p_blue = 0.5 * 0.2 + 0.3 * 0.2 + 0.2 * 0.0
assert math.isclose(joint.xs("black", level="color").sum(), p_black)
assert math.isclose(p_black, 0.31)
assert math.isclose(joint.xs("blue", level="color").sum(), p_blue)
assert math.isclose(p_blue, 0.16)
p_black

0.31000000000000005

## Generalization

The same partition argument gives the marginal of every color and every horsepower value. Each total-probability sum agrees with marginalizing the joint table directly -- two routes, one number, which is the point. Notebook 8 draws this splitting as a tree before notebook 9 inverts it.

In [3]:
color_marginal = joint.groupby(level="color").sum()
horsepower_marginal = joint.groupby(level="horsepower").sum()

# spot checks: p(red) comes only from Ferrari, p(700) only from Porsche
assert math.isclose(color_marginal["red"], 0.2 * 0.6)
assert math.isclose(color_marginal["red"], 0.12)
assert math.isclose(horsepower_marginal.loc[700], 0.3 * 0.05)
assert math.isclose(horsepower_marginal.loc[700], 0.015)
assert math.isclose(color_marginal.sum(), 1.0)
color_marginal

color
black     0.31
blue      0.16
gray      0.34
green     0.03
red       0.12
yellow    0.04
Name: p, dtype: float64

## References

- Blitzstein, J. K., Hwang, J. (2019): "Introduction to Probability", 2nd ed., Chapman & Hall/CRC, chapter "Conditional Probability" (covers the law of total probability), https://www.routledge.com/Introduction-to-Probability-Second-Edition/Blitzstein-Hwang/p/book/9781138369917 (free PDF: https://probabilitybook.net/).
- Wasserman, L. (2004): "All of Statistics: A Concise Course in Statistical Inference", Springer Texts in Statistics, chapter "Probability", https://doi.org/10.1007/978-0-387-21736-9.
- scipy.stats.rv_discrete, https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.rv_discrete.html.